# Exercises

In this section we have one exercise:
1. Implement the polynomial kernel.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [28]:
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        d = 3
        kernel = kernel ** d
    return kernel

In [29]:
from sklearn.datasets import load_iris
import numpy as np
from sklearn.model_selection import train_test_split
import cvxopt

iris = load_iris()
data_set = iris.data
labels = iris.target

data_set = data_set[labels!=2]
labels = labels[labels!=2]

train_data_set, test_data_set, train_labels, test_labels = train_test_split(
    data_set, labels, test_size=0.2, random_state=15)
from sklearn.preprocessing import StandardScaler

# Add normalization for poly kernel: without it, dot products raised to power d. 
# With normalization we have a result 1.0, without it we have a result 0.55

scaler = StandardScaler()
train_data_set = scaler.fit_transform(train_data_set)
test_data_set = scaler.transform(test_data_set)

train_labels[train_labels<1] = -1
test_labels[test_labels<1] = -1

objects_count = len(train_labels)

In [30]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

In [31]:
def classify_poly(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    d = 3
    kernel = np.dot(test_data_set, support_vectors.T) ** d
    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [32]:
lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(train_data_set, train_labels, kernel_type='poly')
predicted = classify_poly(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id)
predicted = list(predicted.astype(int))

from sklearn.metrics import accuracy_score

print(accuracy_score(predicted, test_labels))

     pcost       dcost       gap    pres   dres
 0: -3.0497e+02 -5.9835e+03  1e+04  4e-01  6e-13
 1: -3.0874e+02 -2.2565e+03  2e+03  7e-02  7e-13
 2: -3.6388e+02 -6.5962e+02  3e+02  9e-03  5e-13
 3: -4.3845e+02 -5.1564e+02  8e+01  2e-03  7e-13
 4: -4.5964e+02 -4.8088e+02  2e+01  3e-04  1e-12
 5: -4.6546e+02 -4.7023e+02  5e+00  6e-05  8e-13
 6: -4.6699e+02 -4.6787e+02  9e-01  7e-06  1e-12
 7: -4.6722e+02 -4.6750e+02  3e-01  2e-06  1e-12
 8: -4.6731e+02 -4.6735e+02  4e-02  1e-07  1e-12
 9: -4.6733e+02 -4.6733e+02  9e-04  1e-09  8e-13
10: -4.6733e+02 -4.6733e+02  1e-05  1e-11  8e-13
Optimal solution found.
1.0
